In [21]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.patches import Patch

from subprocess import run, PIPE
import sys
import gc
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [22]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [23]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 1
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 64, number of workers: 1, processes: False


In [24]:
mask = '01'
latitude = [30,60]
months = [5,6,7,8,9,10]
dummy = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/1H/dummy.nc')['PRECT']


In [25]:
print(dummy)

<xarray.DataArray 'PRECT' (time: 1, lat: 192, lon: 288)> Size: 221kB
[55296 values with dtype=float32]
Coordinates:
  * time     (time) object 8B 1850-01-01 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Attributes:
    long_name:     Total (convective and large-scale) precipitation rate (liq...
    units:         m/s
    cell_methods:  time: mean


In [26]:

land_sea = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/24H/mask.nc')['MASK']
land_sea = land_sea[0,:,:]
indexer = ((land_sea.lat>=latitude[0]) & (land_sea.lat<=latitude[1]))
land_sea = land_sea.where(indexer,drop=True)
indexer = ((dummy.lat>=latitude[0]) & (dummy.lat<=latitude[1]))

dummy = dummy.where(indexer,drop=True)
land_sea = land_sea.assign_coords(lat=dummy.lat, lon=dummy.lon)
lat_weights = xr.DataArray(
    np.cos(np.deg2rad(land_sea['lat'])),
    dims=['lat'],
    coords={'lat': land_sea['lat']}
)

In [27]:
print(lat_weights)

<xarray.DataArray 'lat' (lat: 32)> Size: 256B
array([0.86049074, 0.85199496, 0.84326868, 0.83431427, 0.82513415,
       0.8157308 , 0.80610677, 0.79626465, 0.78620712, 0.7759369 ,
       0.76545675, 0.75476953, 0.74387811, 0.73278544, 0.72149454,
       0.71000844, 0.69833026, 0.68646316, 0.67441035, 0.66217508,
       0.64976067, 0.63717049, 0.62440792, 0.61147643, 0.59837951,
       0.58512071, 0.57170362, 0.55813186, 0.54440911, 0.53053907,
       0.5165255 , 0.5023722 ])
Coordinates:
  * lat      (lat) float64 256B 30.63 31.57 32.51 33.46 ... 57.96 58.9 59.84
Attributes:
    standard_name:  latitude
    long_name:      latitude
    units:          degrees_north
    axis:           Y


## IFS deep on

In [ ]:
#IFS deep on land

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/hourly_precip/cesm2_grid')
#files = sorted([str(f) for f in data_path.glob('mask_land_IFS_2.8-FESOM_5_deep*.nc')])[:]

files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_2.8-FESOM_5_deep_on_hourly_precip_??_????.nc')])[:]
ifs_on = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']


In [ ]:

ifs_on = ifs_on.where(ifs_on.time.dt.month.isin(months), drop=True)
indexer = ((ifs_on.lat>=latitude[0]) & (ifs_on.lat<=latitude[1]))

ifs_on = ifs_on.where(indexer,drop=True)
ifs_on = ifs_on.load()

ifs_on_land = ifs_on.where(land_sea > 0, np.nan)
ifs_on_sea = ifs_on.where(land_sea == 0, np.nan)

precip_count_ifs_on_land = ifs_on_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_ifs_on_sea = ifs_on_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [ ]:
precip_count_ifs_on_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_2.8km_on_cesm2_grid_'+mask+'_land_30-60N_precip_count_sum.nc')
precip_count_ifs_on_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_2.8km_on_cesm2_grid_'+mask+'_sea_30-60N_precip_count_sum.nc')


## IFS deep off

In [ ]:
#IFS deep off land

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/hourly_precip/cesm2_grid')
#files = sorted([str(f) for f in data_path.glob('mask_land_IFS_2.8-FESOM_5_deep*.nc')])[:]

files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_2.8-FESOM_5_deep_off_hourly_precip_??_????.nc')])[:]
ifs_off = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']


In [ ]:

ifs_off = ifs_off.where(ifs_off.time.dt.month.isin(months), drop=True)
indexer = ((ifs_off.lat>=latitude[0]) & (ifs_off.lat<=latitude[1]))

ifs_off = ifs_off.where(indexer,drop=True)
ifs_off = ifs_off.load()

ifs_off_land = ifs_off.where(land_sea > 0, np.nan)
ifs_off_sea = ifs_off.where(land_sea == 0, np.nan)

precip_count_ifs_off_land = ifs_off_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_ifs_off_sea = ifs_off_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [ ]:
precip_count_ifs_off_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_2.8km_off_cesm2_grid_'+mask+'_land_30-60N_precip_count_sum.nc')
precip_count_ifs_off_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_2.8km_off_cesm2_grid_'+mask+'_sea_30-60N_precip_count_sum.nc')


## IFS cycle3

In [ ]:
#IFS cycle3 land

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/hourly_precip/cesm2_grid')

files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_4.4-FESOM_5_precip_cycle3_??_????.nc')])[:]
ifs_cycle3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']


In [ ]:

ifs_cycle3 = ifs_cycle3.where(ifs_cycle3.time.dt.month.isin(months), drop=True)
indexer = ((ifs_cycle3.lat>=latitude[0]) & (ifs_cycle3.lat<=latitude[1]))

ifs_cycle3 = ifs_cycle3.where(indexer,drop=True)
ifs_cycle3 = ifs_cycle3.load()

ifs_cycle3_land = ifs_cycle3.where(land_sea > 0, np.nan)
ifs_cycle3_sea = ifs_cycle3.where(land_sea == 0, np.nan)

precip_count_ifs_cycle3_land = ifs_cycle3_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_ifs_cycle3_sea = ifs_cycle3_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [ ]:
precip_count_ifs_cycle3_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle3_cesm2_grid_'+mask+'_land_30-60N_precip_count_sum.nc')
precip_count_ifs_cycle3_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle3_cesm2_grid_'+mask+'_sea_30-60N_precip_count_sum.nc')


## IFS cycle4 historical

In [ ]:
#IFS cycle4 land

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/1990-2020/hourly_precip/cesm2_grid')

files = sorted([str(f) for f in data_path.glob('mask_01_cesm2_interp_higher_IFS_9-FESOM_5_hourly_precip_??_????.nc')])[:]
ifs_cycle4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']


In [ ]:

ifs_cycle4 = ifs_cycle4.where(ifs_cycle4.time.dt.month.isin(months), drop=True)
indexer = ((ifs_cycle4.lat>=latitude[0]) & (ifs_cycle4.lat<=latitude[1]))

ifs_cycle4 = ifs_cycle4.where(indexer,drop=True)
ifs_cycle4 = ifs_cycle4.load()

ifs_cycle4_land = ifs_cycle4.where(land_sea > 0, np.nan)
ifs_cycle4_sea = ifs_cycle4.where(land_sea == 0, np.nan)

precip_count_ifs_cycle4_land = ifs_cycle4_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_ifs_cycle4_sea = ifs_cycle4_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [ ]:
precip_count_ifs_cycle4_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle4_1990-2019_cesm2_grid_higher_'+mask+'_land_30-60N_precip_count_sum.nc')
precip_count_ifs_cycle4_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle4_1990-2019_cesm2_grid_higher_'+mask+'_sea_30-60N_precip_count_sum.nc')


## IFS cycle4 future

In [ ]:
years = np.arange(2020,2050,1)
for year in years:
    print(year)
    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/hourly_precip/cesm2_grid')

    files = sorted([str(f) for f in data_path.glob('mask_01_cesm2_interp_IFS_9-FESOM_5_hourly_precip_??_'+str(year)+'.nc')])[:]
    ifs_cycle4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']

    ifs_cycle4 = ifs_cycle4.where(ifs_cycle4.time.dt.month.isin(months), drop=True)
    indexer = ((ifs_cycle4.lat>=latitude[0]) & (ifs_cycle4.lat<=latitude[1]))

    ifs_cycle4 = ifs_cycle4.where(indexer,drop=True)
    ifs_cycle4 = ifs_cycle4.load()

    ifs_cycle4_land = ifs_cycle4.where(land_sea > 0, np.nan)
    ifs_cycle4_sea = ifs_cycle4.where(land_sea == 0, np.nan)

    precip_count_ifs_cycle4_land = ifs_cycle4_land.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_ifs_cycle4_sea = ifs_cycle4_sea.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_ifs_cycle4_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle4_'+str(year)+'_cesm2_grid_'+mask+'_land_30-60N_precip_count_sum.nc')
    precip_count_ifs_cycle4_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/ifs_cycle4_'+str(year)+'_cesm2_grid_'+mask+'_sea_30-60N_precip_count_sum.nc')


## ICON cycle3

In [18]:
#icon cycle3 land

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/hourly_precip/cesm2_grid')

files = sorted([str(f) for f in data_path.glob('mask_01_cesm2_interp_higher_ICON_ngc3028_hourly_precip_??_????.nc')])[:]
icon_cycle3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']


In [19]:

icon_cycle3 = icon_cycle3.where(icon_cycle3.time.dt.month.isin(months), drop=True)
indexer = ((icon_cycle3.lat>=latitude[0]) & (icon_cycle3.lat<=latitude[1]))

icon_cycle3 = icon_cycle3.where(indexer,drop=True)
icon_cycle3 = icon_cycle3.load()

icon_cycle3_land = icon_cycle3.where(land_sea > 0, np.nan)
icon_cycle3_sea = icon_cycle3.where(land_sea == 0, np.nan)

precip_count_icon_cycle3_land = icon_cycle3_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_icon_cycle3_sea = icon_cycle3_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [20]:
precip_count_icon_cycle3_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/icon_cycle3_cesm2_grid_higher_'+mask+'_land_30-60N_precip_count_sum.nc')
precip_count_icon_cycle3_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/icon_cycle3_cesm2_grid_higher_'+mask+'_sea_30-60N_precip_count_sum.nc')


## ICON cycle4

In [28]:
years = np.arange(2020,2021,1)
for year in years:
    print(year)
    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/hourly_precip/cesm2_grid')

    files = sorted([str(f) for f in data_path.glob('mask_01_cesm2_interp_higher_ICON_ngc4008_hourly_precip_??_'+str(year)+'.nc')])[:]
    icon_cycle4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']

    icon_cycle4 = icon_cycle4.where(icon_cycle4.time.dt.month.isin(months), drop=True)
    indexer = ((icon_cycle4.lat>=latitude[0]) & (icon_cycle4.lat<=latitude[1]))

    icon_cycle4 = icon_cycle4.where(indexer,drop=True)
    icon_cycle4 = icon_cycle4.load()

    icon_cycle4_land = icon_cycle4.where(land_sea > 0, np.nan)
    icon_cycle4_sea = icon_cycle4.where(land_sea == 0, np.nan)

    precip_count_icon_cycle4_land = icon_cycle4_land.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_icon_cycle4_sea = icon_cycle4_sea.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_icon_cycle4_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/icon_cycle4_'+str(year)+'_cesm2_grid_higher_'+mask+'_land_30-60N_precip_count_sum.nc')
    precip_count_icon_cycle4_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/icon_cycle4_'+str(year)+'_cesm2_grid_higher_'+mask+'_sea_30-60N_precip_count_sum.nc')


2020


## IMERG

In [ ]:
years = np.arange(2024,2025,1)
for year in years:
    print(year)
    data_path = Path('/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/cesm2_grid')

    files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IMERG_V07B_final_??_'+str(year)+'.nc4')])[:]
    imerg = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']

    imerg = imerg.where(imerg.time.dt.month.isin(months), drop=True)
    indexer = ((imerg.lat>=latitude[0]) & (imerg.lat<=latitude[1]))

    imerg = imerg.where(indexer,drop=True)
    imerg = imerg.load()

    imerg_land = imerg.where(land_sea > 0, np.nan)
    imerg_sea = imerg.where(land_sea == 0, np.nan)

    precip_count_imerg_land = imerg_land.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_imerg_sea = imerg_sea.weighted(lat_weights).sum(dim=['lat','lon'])
    precip_count_imerg_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_grid/imerg_'+str(year)+'_cesm2_grid_'+mask+'_land_30-60N_precip_count_sum.nc')
    precip_count_imerg_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_grid/imerg_'+str(year)+'_cesm2_grid_'+mask+'_sea_30-60N_precip_count_sum.nc')


## CESM2

In [ ]:
cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/1H/b.e21.BHISTsmbb.f09_g17.LE2-1011.001.cam.h4.30-60N_PRECT.1850010100-2014123100.nc')['PRECT']
cesm2 = cesm2 * 3600 * 1000
months = [1,2,3,4,11,12]
#months = [5,6,7,8,9,10]

cesm2 = cesm2.where(cesm2.time.dt.month.isin(months), drop=True)

In [ ]:
latitude = [-50,-20] #current standard
#latitude = [30,60]

if mask == '01':
    cesm2 = xr.where(cesm2 > 0.1, 1, np.nan)
elif mask == '02':
    cesm2 = xr.where(cesm2 > 0.2, 1, np.nan)
elif mask == '05':
    cesm2 = xr.where(cesm2 > 0.5, 1, np.nan)

lat_weights = xr.DataArray(
    np.cos(np.deg2rad(cesm2['lat'])),
    dims=['lat'],
    coords={'lat': cesm2['lat']}
)

In [ ]:
land_sea_cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/24H/mask.nc')['MASK']
land_sea_cesm2 = land_sea_cesm2[0,:,:]
indexer = ((land_sea_cesm2.lat>=latitude[0]) & (land_sea_cesm2.lat<=latitude[1]))
land_sea_cesm2 = land_sea_cesm2.where(indexer,drop=True)
land_sea_cesm2 = land_sea_cesm2.assign_coords(lat=cesm2.lat, lon=cesm2.lon)
cesm2_land = cesm2.where(land_sea_cesm2 > 0, np.nan)
cesm2_sea = cesm2.where(land_sea_cesm2 == 0, np.nan)

In [ ]:
precip_count_cesm2_land = cesm2_land.weighted(lat_weights).sum(dim=['lat','lon'])
precip_count_cesm2_sea = cesm2_sea.weighted(lat_weights).sum(dim=['lat','lon'])

In [ ]:
#precip_count_cesm2_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_land_'+mask+'_30-60N_precip_count_sum.nc')
#precip_count_cesm2_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_sea_'+mask+'_30-60N_precip_count_sum.nc')


In [ ]:
precip_count_cesm2_land.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_land_'+mask+'_20-50S_precip_count_sum.nc')
precip_count_cesm2_sea.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/cesm2_sea_'+mask+'_20-50S_precip_count_sum.nc')
